# Insider Trading ETL Demo

Run the pipeline, inspect loaded transactions, then apply a simple signal filter.

In [ ]:
import psycopg2
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 60)
pd.set_option('display.float_format', '{:,.2f}'.format)

def get_conn():
    return psycopg2.connect(dbname="insider_signals", user="pablolasarte")

print("Connected to insider_signals")

## 1. Run the pipeline

In [ ]:
!python etl.py

print("\nLatest pipeline log output:\n")
!tail -n 25 pipeline.log

## 2. Query the loaded transactions

In [ ]:
transactions_query = """
    SELECT
        f.accession_number,
        i.ticker,
        i.name AS issuer,
        ins.name AS reporting_owner,
        f.filing_date,
        t.transaction_code,
        t.security_title,
        t.shares,
        t.price,
        ROUND((t.shares * t.price)::numeric, 2) AS total_value,
        t.disposed_code,
        f.aff10b5One AS is_10b5_plan
    FROM transactions t
    JOIN filing f ON t.accession_number = f.accession_number
    JOIN issuer i ON f.issuer_cik = i.issuer_cik
    JOIN insider ins ON f.insider_cik = ins.insider_cik
    ORDER BY f.filing_date DESC, total_value DESC NULLS LAST
    LIMIT 25;
"""

with get_conn() as conn:
    df_transactions = pd.read_sql(transactions_query, conn)

print(f"Loaded transaction sample: {len(df_transactions)} rows shown")
df_transactions

## 3. Signal filter: purchases outside 10b5-1 plans

In [ ]:
signal_query = """
    SELECT
        i.ticker,
        i.name AS issuer,
        ins.name AS reporting_owner,
        f.filing_date,
        t.transaction_code,
        t.security_title,
        t.shares AS shares_bought,
        t.price,
        ROUND((t.shares * t.price)::numeric, 2) AS total_value,
        f.aff10b5One AS is_10b5_plan
    FROM transactions t
    JOIN filing f ON t.accession_number = f.accession_number
    JOIN issuer i ON f.issuer_cik = i.issuer_cik
    JOIN insider ins ON f.insider_cik = ins.insider_cik
    WHERE t.transaction_code = 'P'
      AND f.aff10b5One = FALSE
    ORDER BY f.filing_date DESC, total_value DESC NULLS LAST;
"""

with get_conn() as conn:
    df_signals = pd.read_sql(signal_query, conn)

print(f"P transactions with aff10b5One = FALSE: {len(df_signals)} rows")
df_signals